In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_SEED = 1000
df = pl.read_parquet("data/1L83:p2rank:1.fingerprints.parquet")

In [3]:
# df = df[:10000]

In [ ]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x_scalars = df.select(FEATURE_NAMES).to_numpy()

x_morgan = np.array(df["morgan_fingerprint"].to_list())
x_e3fp = np.array(df["e3fp"].to_list())
x_usrcat = np.array(df["usrcat"].to_list())
x_whim = np.array(df["whim"].to_list())
x_getaway = np.array(df["getaway"].to_list())
x_pharmacophore = np.array(df["pharmacophore_3d"].to_list())

x_all_fingerprints = np.hstack(
    [
        x_morgan,
        x_e3fp,
        # x_usrcat,
        # x_whim,
        x_getaway,
        x_pharmacophore,
    ]
)

x = np.hstack([x_scalars, x_all_fingerprints])

y = df[LABEL_NAME].to_numpy()

In [ ]:
from surrogate_model.optuna import make_objective

x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.8, random_state=RANDOM_SEED
)

NUM_TRIALS = 100

PRIMARY_METRIC = "bedroc"

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    make_objective(
        X=x_train,
        y=y_train,
        n_splits=3,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

In [ ]:
all_feature_names = list(FEATURE_NAMES)
fingerprint_arrays = {
    "morgan": x_morgan,
    "e3fp": x_e3fp,
    "usrcat": x_usrcat,
    "whim": x_whim,
    "getaway": x_getaway,
    "pharmacophore_3d": x_pharmacophore,
}

for prefix, arr in fingerprint_arrays.items():
    all_feature_names.extend([f"{prefix}_{i}" for i in range(arr.shape[1])])

In [ ]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params, deterministic=True, force_row_wise=True)
final_model.fit(
    x_train,
    y_train,
    eval_X=x_test,
    eval_y=y_test,
    feature_name=all_feature_names,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

In [ ]:
feature_importance_df = (
    pl.DataFrame(
        {
            "feature": final_model.feature_name_,
            "gain": final_model.booster_.feature_importance(importance_type="gain"),
        }
    )
    .with_columns(importance_pct=(pl.col("gain") / pl.col("gain").sum()) * 100)
    .sort("importance_pct", descending=True)
)

In [ ]:
feature_importance_df.head(10)

In [ ]:
from skfp.metrics import bedroc_score, enrichment_factor, spearman_correlation

y_pred = np.asarray(final_model.predict(x_test))

active_quantile = 0.05
affinity_threshold = np.quantile(y_test, active_quantile)
y_true_binary = (y_test <= affinity_threshold).astype(int)
scores = -y_pred

results = {
    "spearman": float(spearman_correlation(y_test, y_pred)),
    "bedroc": float(bedroc_score(y_true_binary, scores, alpha=20.0)),
    "enrichment_factor_1_percent": float(
        enrichment_factor(y_true_binary, scores, 0.01)
    ),
    "enrichment_factor_5_percent": float(
        enrichment_factor(y_true_binary, scores, 0.05)
    ),
    "enrichment_factor_10_percent": enrichment_factor(y_true_binary, scores, 0.1),
}

results

| dataset size | trials | fingerprints | train time | spearman | bedroc | EF@1%| EF@5% | EF@10% |
| - | - | - | - | - | - | - | - | - |
| 100000 | 50 | morgan | 3:35 | 0.461 | 0.323 | 4.91 | 4.82 | 3.96 | 
| 100000 | 100 | morgan | 6:36 | 0.463 | 0.315 | 4.26 | 4.84 | 3.84 | 
| 100000 | 50 | e3fp | 4:49 | 0.280 | 0.291 | 7.96 | 4.26 | 3.08 |
| 100000 | 100 | e3fp | 11:15 | 0.281 | 0.294 | 7.78 | 4.52 | 3.05 |
| 100000 | 100 | e3fp, morgan | | | | | | |
| 100000 | 100 | all | 2:19:30 | 0.475 | 0.335 | 6.39 | 5.04 | 3.86 |


Targets:

Spearman ρ > 0.5

EF@1% > 10-20 is considered reasonably strong for early enrichment in virtual screening
EF@1% > 30-50 is very good, approaching what you'd see from decent docking scores themselves
